In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


# 1. 데이터 가져오기

In [20]:
import pandas as pd
df = pd.read_csv(r'C:\ai_x\download\shareData\부동산_250213\최종전국평당분양가격(결측치제외).csv',
         encoding='cp949')

In [3]:
df.head()

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0


# 2. 지역명의 라벨인코딩
- 라벨인코딩한 지역명2 필드를 추가 / 제대로 안돌아갈 확률이 높음
- 이 후 원핫인코딩을 해보자

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [23]:
df_input = df.iloc[:,:-1]

In [32]:
df['지역명2'] = le.fit_transform(df['지역명'])
X_data = df[['지역명2', '연도', '월']].values
y_data = df[['평당분양가격']].values

In [35]:
X_data,y_data

(array([[   8, 2013,   12],
        [   7, 2013,   12],
        [   5, 2013,   12],
        ...,
        [   3, 2024,    8],
        [   2, 2024,    8],
        [  14, 2024,    8]], dtype=int64),
 array([[18189. ],
        [ 8111. ],
        [ 8080. ],
        ...,
        [13827. ],
        [13252.8],
        [25419.9]]))

# 3. 변수간 스케일 조정(min-max / std)
- 입력변수와 타겟 변수 따로 스케일 조정
- normalization : 지역명2n, 연도n, 월n 필드 추가
- standardization : 지역명2s, 연도s, 월s 필드 추가

In [34]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [36]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaled_X_data = scaler_X.fit_transform(X_data)
scaled_y_data = scaler_y.fit_transform(y_data)

In [42]:
df_n = pd.DataFrame({'지역명2n' : [x[0] for x in scaled_X_data],
                    '연도n' : [x[1] for x in scaled_X_data],
                    '월n' : [x[2] for x in scaled_X_data] })
df_n.head()

,지역명2n,연도n,월n
0,0.5000,0.0,1.0
1,0.4375,0.0,1.0
2,0.3125,0.0,1.0
3,0.6875,0.0,1.0
4,0.2500,0.0,1.0


In [41]:
df = pd.concat([df, df_n], axis=1)
df.head()

,지역명,연도,월,평당분양가격,지역명2,지역명2n,연도n,월n,지역명2n,연도n,월n
0,서울,2013,12,18189.0,8,0.5000,0.0,1.0,0.5000,0.0,1.0
1,부산,2013,12,8111.0,7,0.4375,0.0,1.0,0.4375,0.0,1.0
2,대구,2013,12,8080.0,5,0.3125,0.0,1.0,0.3125,0.0,1.0
3,인천,2013,12,10204.0,11,0.6875,0.0,1.0,0.6875,0.0,1.0
4,광주,2013,12,6098.0,4,0.2500,0.0,1.0,0.2500,0.0,1.0


In [50]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

Instructions for updating:
non-resource variables are not supported in the long term


In [57]:
# placeholder 설정 (입력변수 x, 타겟변수 y를 나중에)
X = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

# W, b
W = tf.Variable(tf.random.normal([1]), name = 'weight')
b = tf.Variable(tf.random.normal([1]), name = 'bias')

# Hypothesis(예측값)
H = W*X + b

# 손실함수
cost = tf.reduce_mean(tf.square(H-y))

# 경사하강법
train = tf.train.GradientDescentOptimizer(learning_rate=0.001).minimize(cost)

# 세션 객체 생성
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # W, b 초기화

# n번 학습
for step in range(30001):
    _, cost_val, W_val, b_val = sess.run([train, cost, W, b], feed_dict={X:scaled_X_data, 
                                                                         y:scaled_y_data})
    if step%3000 ==0:
        print("{}째 : cost : {}, W:{}, b : {}".format(step,
                                                    cost_val,
                                                    W_val,
                                                    b_val))

0째 : cost : 0.6255365014076233, W:[-0.8084382], b : [-0.15482455]
3000째 : cost : 0.0308570247143507, W:[-0.29437143], b : [0.32911587]
6000째 : cost : 0.02393028326332569, W:[-0.16865131], b : [0.26085928]
9000째 : cost : 0.0209855567663908, W:[-0.08677695], b : [0.21617526]
12000째 : cost : 0.01973365619778633, W:[-0.03339299], b : [0.18704018]
15000째 : cost : 0.019201433286070824, W:[0.0014145], b : [0.16804346]
18000째 : cost : 0.018975166603922844, W:[0.02410965], b : [0.1556573]
21000째 : cost : 0.01887897402048111, W:[0.03890739], b : [0.14758122]
24000째 : cost : 0.018838081508874893, W:[0.04855576], b : [0.14231549]
27000째 : cost : 0.01882069557905197, W:[0.05484664], b : [0.1388821]
30000째 : cost : 0.018813304603099823, W:[0.05894832], b : [0.1366438]
